In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 4 (Court Judgment Prediction): Architecture-3 E-LegalFormer (frozen) + LoRMA
#
#   document -> its LAST 1536 tokens -> 3 chunks of 512 tokens
#            -> every chunk through the E-LegalFormer -> one vector per chunk
#            -> BiLSTM over the chunks + attention pooling -> 0 / 1
#
# Trained on TRAIN only; dev picks the best epoch and the decision threshold.
# ============================================================

RUN_NAME = "p4_arch3_lorma"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-3 checkpoint. If wrong, CELL 10 searches /kaggle/input for it.
CHECKPOINT_PATH = Path(
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-3/"
    "model3_checkpoints/best_model.pt"
)

# Phase-4 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-4-cjpe/Phase_4_CJPE")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Which part of the (long) document the model reads
#   "end"       = the last DOC_TOKENS tokens (closest to the decision; default)
#   "start"     = the first DOC_TOKENS tokens (facts, background)
#   "start_end" = half from the start + half from the end
# ------------------------------------------------------------

DOC_PART = "end"
DOC_TOKENS = 1536            # tokens per document the model reads
CHUNK_LEN = 512              # every chunk goes through the E-LegalFormer separately


# ------------------------------------------------------------
# LoRMA (Bihany, Patel & Modi, 2025):  W' = (I + alpha/r * B A) W
#   - Phase-1 weights W stay FROZEN
#   - only the small matrices A, B are trained (B starts at 0 -> W' = W)
# ------------------------------------------------------------

# every linear layer of every block
LORMA_TARGETS = ["attention.q_proj", "attention.k_proj", "attention.v_proj", "attention.o_proj",
                 "ffn.gate_proj", "ffn.up_proj", "ffn.down_proj"]
LORMA_RANK = 16
LORMA_ALPHA = 32
LORMA_DROPOUT = 0.05

LR_PEFT = 1e-4              # small: LoRMA changes the features the head reads


# ------------------------------------------------------------
# Head:
#   token vectors -> STANDARDIZED (per feature, statistics of the Phase-1 model, CELL 12)
#   chunk vector  = [attention-pooled ; mean] of its token vectors, STANDARDIZED again
#                   (the attention LEARNS which tokens matter, e.g. "dismissed", "set aside")
#   case          = BiLSTM over the chunks -> attention pooling -> 2 classes
# ------------------------------------------------------------

POOL_SKIP_TOKENS = 4         # the first tokens of a chunk are not pooled (they behave abnormally in LMs)
LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.1
LR_HEAD = 1e-3

CLASS_BALANCING = True       # loss weights = inverse class frequency (macro-F1 counts both classes)


# ------------------------------------------------------------
# Training (TRAIN data only)
# ------------------------------------------------------------

EPOCHS = 10
EPOCH_DOCS = 8000            # documents per epoch (all train documents are used before any repeats)
BATCH_DOCS = 4               # documents per forward pass (4 x 3 chunks x 512 tokens). If CUDA out-of-memory: 2
GRAD_ACCUM = 4               # -> 16 documents per weight update
PATIENCE = 2                 # stop if dev macro-F1 does not improve for 2 epochs
HEAD_ONLY_EPOCHS = 1         # epoch 1 trains only the head (LoRMA waits). Its dev score =
                             # frozen Phase-1 model + head: the baseline that LoRMA must beat
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0

EVAL_CHUNKS = 48             # chunks per forward pass when predicting

# Before training (CELL 12): token statistics from STATS_DOCS train cases, and a
# linear probe on the FROZEN Phase-1 features of PROBE_DOCS train cases (~6 minutes).
STATS_DOCS = 1000
PROBE_DOCS = 3000

# CELL 8: where is the signal? A plain word-count model (TF-IDF) on the start, the
# end and the whole case. Data analysis only (~10 minutes). It does not use the model, so
# it gives the same numbers in every Phase-4 notebook: off here, set True to run it.
RUN_SIGNAL_CHECK = False
SIGNAL_TRAIN_DOCS = 15000


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 64              # per split

if DEBUG_MODE:
    EPOCHS = 2
    EPOCH_DOCS = 64
    PROBE_DOCS = 64
    STATS_DOCS = 64
    SIGNAL_TRAIN_DOCS = 64


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert DOC_PART in ("end", "start", "start_end")
assert DOC_TOKENS % CHUNK_LEN == 0

MAX_CHUNKS = DOC_TOKENS // CHUNK_LEN

print("Run       :", RUN_NAME)
print("Reads     :", f"{DOC_PART} of each document, {DOC_TOKENS} tokens = {MAX_CHUNKS} chunks x {CHUNK_LEN}")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p4_arch3_lorma
Reads     : end of each document, 1536 tokens = 3 chunks x 512
Output    : /kaggle/working/p4_arch3_lorma
AMP dtype : torch.float16


In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Documents are long, so only the part we need is tokenized
# (e.g. the last ~DOC_TOKENS tokens), and every pre-token piece
# is BPE-encoded only once (cache).
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


def head_tokens(text, n):
    """First n tokens of text (tokenizes only the beginning, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[:n]

        part = text[:chars]
        part = part[:max(part.rfind(" "), 1)]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[:n]

        chars *= 2


def tail_tokens(text, n):
    """Last n tokens of text (tokenizes only the end, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[-n:]

        part = text[-chars:]
        part = part[max(part.find(" "), 0):]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[-n:]

        chars *= 2


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
print("Demo:", [tokenizer.id_to_token[i] for i in encode_fast(demo)][:12])

Vocabulary size: 32004 | PAD / UNK: 0 1
Demo: ['The', ' ', 'appeal', ' ', 'is', ' ', 'allow', 'ed', '.', ' ', 'The', ' ']


In [4]:
# ============================================================
# CELL 4 — PHASE-4 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev":   "Phase_4_CJPE/Train/dev.json",
    "test":  "Phase_4_CJPE/Test/test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE4_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_4_CJPE/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE4_FILES.items():

        DATA_FILES[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )

for name, path in DATA_FILES.items():
    print(f"{name:6s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")

train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE/Train/train.json  (737 MB)
dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE/Train/dev.json  (80 MB)
test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE/Test/test.json  (35 MB)


In [5]:
# ============================================================
# CELL 5 — LOAD CASES + STATISTICS
#
# Every case: {"id", "text", "label" (0 / 1, or None for test)}
# ============================================================

LABEL_KEYS = ["label", "labels", "judgment", "decision", "y"]


def read_json(path):
    """A JSON file, or a JSON-lines file (one case per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_cases(path, with_labels=True):

    raw = read_json(path)

    # accepted layouts: [case, ...]  |  {"data": [case, ...]}  |  {case_id: case}
    if isinstance(raw, dict):

        list_key = next((k for k in ("data", "cases", "documents") if isinstance(raw.get(k), list)), None)

        if list_key is not None:
            raw = raw[list_key]
        else:
            raw = [dict(value, id=key) if isinstance(value, dict) else {"id": key, "text": value}
                   for key, value in raw.items()]

    cases = []

    for i, item in enumerate(raw):

        label_key = next((k for k in LABEL_KEYS if k in item), None) if with_labels else None
        label = item[label_key] if label_key is not None else None

        if isinstance(label, list):                 # e.g. [1]
            label = label[0]

        cases.append({
            "id": str(item.get("id", i)),
            "text": str(item.get("text", "")),
            "label": None if label is None else int(label),
        })

    if DEBUG_MODE:
        cases = cases[:DEBUG_DOCS]

    return cases


train_cases = load_cases(DATA_FILES["train"])
dev_cases = load_cases(DATA_FILES["dev"])
test_cases = load_cases(DATA_FILES["test"], with_labels=False)     # test labels are private: never read

for name, cases in [("train", train_cases), ("dev", dev_cases)]:
    assert all(c["label"] in (0, 1) for c in cases), f"{name}: labels must be 0 or 1"

ids = [c["id"] for c in test_cases]
assert len(ids) == len(set(ids)), "test ids are not unique"


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(f"{'split':6s} {'cases':>7s} {'label 0':>8s} {'label 1':>8s} {'chars (median / 90% / max)':>30s}")

for name, cases in [("train", train_cases), ("dev", dev_cases), ("test", test_cases)]:

    labels = Counter(c["label"] for c in cases)
    chars = np.array([len(c["text"]) for c in cases])

    print(f"{name:6s} {len(cases):7d} {labels.get(0, 0):8d} {labels.get(1, 0):8d} "
          f"{np.median(chars):12.0f} / {np.percentile(chars, 90):8.0f} / {chars.max():8d}")

empty = sum(not c["text"].strip() for c in train_cases + dev_cases + test_cases)
print("\nEmpty texts:", empty)

print("\nSample:", train_cases[0]["id"], "| label", train_cases[0]["label"])
print("  start:", repr(train_cases[0]["text"][:200]))
print("  end  :", repr(train_cases[0]["text"][-200:]))


# ------------------------------------------------------------
# Class weights for the loss (inverse frequency, mean 1)
# ------------------------------------------------------------

train_label_counts = Counter(c["label"] for c in train_cases)

if CLASS_BALANCING:
    CLASS_WEIGHTS = torch.tensor([len(train_cases) / (2 * max(train_label_counts[k], 1)) for k in (0, 1)])
else:
    CLASS_WEIGHTS = torch.ones(2)

print("\nClass weights:", [round(w, 3) for w in CLASS_WEIGHTS.tolist()])

split    cases  label 0  label 1     chars (median / 90% / max)
train    37387    22067    15320        13023 /    38996 /  2492701
dev       3505     1749     1756        16988 /    41909 /   260198
test      1517        0        0        17320 /    42859 /   233000

Empty texts: 0

Sample: 2020_1 | label 0

Class weights: [0.847, 1.22]


In [6]:
# ============================================================
# CELL 6 — TOKENIZE (only the part the model reads) + CHUNKS
#
# DOC_PART = "end": the last DOC_TOKENS tokens, cut into chunks of
# CHUNK_LEN FROM THE END, so the last chunk always ends where the
# document ends.
# ============================================================

def split_chunks(ids, from_end):

    if from_end:
        return [ids[max(0, i - CHUNK_LEN):i] for i in range(len(ids), 0, -CHUNK_LEN)][::-1]

    return [ids[i:i + CHUNK_LEN] for i in range(0, len(ids), CHUNK_LEN)]


def make_chunks(text):
    """List of token-id arrays (at most MAX_CHUNKS, each at most CHUNK_LEN)."""

    if DOC_PART == "end":
        chunks = split_chunks(tail_tokens(text, DOC_TOKENS), from_end=True)

    elif DOC_PART == "start":
        chunks = split_chunks(head_tokens(text, DOC_TOKENS), from_end=False)

    else:  # start_end
        half = DOC_TOKENS // 2
        start = head_tokens(text, half)
        end = tail_tokens(text, half)

        if len(start) < half or len(end) < half:          # short document: read all of it once
            chunks = split_chunks(head_tokens(text, DOC_TOKENS), from_end=False)
        else:
            chunks = split_chunks(start, from_end=False) + split_chunks(end, from_end=True)

    chunks = [c for c in chunks if len(c) > 0] or [[UNK_ID]]      # empty text -> 1 token

    return [np.array(c, dtype=np.int32) for c in chunks]


t0 = time.time()

for split, cases in [("train", train_cases), ("dev", dev_cases), ("test", test_cases)]:

    for k, case in enumerate(cases):

        case["chunks"] = make_chunks(case["text"])
        case["n_tokens"] = sum(len(c) for c in case["chunks"])

        if (k + 1) % 5000 == 0:
            print(f"  {split}: {k + 1}/{len(cases)} cases | {time.time() - t0:.0f}s", flush=True)

print(f"Tokenized in {time.time() - t0:.0f}s")


# check: fast encoding == tokenizer.encode on a few real cases
for case in train_cases[:5]:
    piece = case["text"][-3000:]
    assert encode_fast(piece) == tokenizer.encode(piece)


n_tokens = np.array([c["n_tokens"] for c in train_cases])
print(f"\nTrain cases shorter than {DOC_TOKENS} tokens: {(n_tokens < DOC_TOKENS).mean():.1%}")
print(f"Chunks per case: {Counter(len(c['chunks']) for c in train_cases)}")
print(f"Tokens per epoch: {EPOCH_DOCS * DOC_TOKENS / 1e6:.1f}M "
      f"({min(EPOCH_DOCS, len(train_cases))} cases)")

  train: 5000/37387 cases | 9s
  train: 10000/37387 cases | 16s
  train: 15000/37387 cases | 23s
  train: 20000/37387 cases | 31s
  train: 25000/37387 cases | 39s
  train: 30000/37387 cases | 45s
  train: 35000/37387 cases | 53s
Tokenized in 65s

Train cases shorter than 1536 tokens: 15.5%
Chunks per case: Counter({3: 33517, 2: 2117, 1: 1753})
Tokens per epoch: 12.3M (8000 cases)


In [7]:
# ============================================================
# CELL 7 — METRICS + DECISION THRESHOLD
# ============================================================

def binary_scores(labels, preds):
    """Macro P / R / F1 over the 2 classes (the competition metric is macro-F1)."""

    p, r, f, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1], average="macro", zero_division=0)

    return {"precision": float(p), "recall": float(r), "f1": float(f)}


def best_threshold(labels, probs):
    """Threshold on P(label = 1) that gives the best macro-F1."""

    best_t, best_f1 = 0.5, -1.0

    # fixed grid + quantiles of the probabilities (works even when they are all close together)
    candidates = np.unique(np.concatenate([np.arange(0.05, 0.951, 0.01), np.quantile(probs, np.linspace(0.02, 0.98, 97))]))

    for t in candidates:

        f1 = f1_score(labels, (probs >= t).astype(int), labels=[0, 1], average="macro", zero_division=0)

        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)

    return best_t, best_f1


_labels = np.array([0, 0, 1, 1])
_probs = np.array([0.2, 0.6, 0.7, 0.9])
print("check:", binary_scores(_labels, (_probs >= 0.5).astype(int)), "| best threshold:", best_threshold(_labels, _probs))

In [8]:
# ============================================================
# CELL 8 — WHERE IS THE SIGNAL?  (data analysis only, NOT a submission)
#
# A plain word-count model (TF-IDF + logistic regression, trained on
# TRAIN, scored on dev) on 3 parts of every case:
#   start  = the first N characters
#   end    = the last N characters
#   whole  = the whole case (first 300,000 characters)
# N = the length of the text the model reads (DOC_PART, DOC_TOKENS).
# It uses no pretrained model; it only shows WHERE in a judgment the
# outcome is visible and what score plain word counts reach.
# ============================================================

signal_check = {}

if RUN_SIGNAL_CHECK:

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    t0 = time.time()

    signal_train = random.Random(SEED).sample(train_cases, min(SIGNAL_TRAIN_DOCS, len(train_cases)))

    def read_length(case):
        # characters of the text the model reads
        return len(tokenizer.decode([int(t) for chunk in case["chunks"] for t in chunk]))

    def parts(case):
        n = read_length(case)
        return {"start": case["text"][:n], "end": case["text"][-n:], "whole": case["text"][:300_000]}

    train_parts = [parts(c) for c in signal_train]
    dev_parts = [parts(c) for c in dev_cases]

    y_train = np.array([c["label"] for c in signal_train])
    y_dev = np.array([c["label"] for c in dev_cases])

    print(f"{'part':8s} {'dev AUC':>8s} {'macro-F1':>9s} {'threshold':>10s}")

    for part in ["start", "end", "whole"]:

        vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=200_000,
                                     sublinear_tf=True, dtype=np.float32)

        X_train = vectorizer.fit_transform([p[part] for p in train_parts])
        X_dev = vectorizer.transform([p[part] for p in dev_parts])

        clf = LogisticRegression(C=4.0, solver="liblinear", class_weight="balanced")
        clf.fit(X_train, y_train)
        p_dev = clf.predict_proba(X_dev)[:, 1]

        t, f1 = best_threshold(y_dev, p_dev)
        signal_check[part] = {"auc": float(roc_auc_score(y_dev, p_dev)), "f1": f1}

        print(f"{part:8s} {signal_check[part]['auc']:8.4f} {f1:9.4f} {t:10.2f}  ({time.time() - t0:.0f}s)", flush=True)

    del train_parts, dev_parts, X_train, X_dev

else:
    print("Signal check skipped (RUN_SIGNAL_CHECK = False)")

Signal check skipped (RUN_SIGNAL_CHECK = False)


In [9]:
# ============================================================
# CELL 9 — MODEL
#
#   every chunk -> Architecture-3 E-LegalFormer (Phase-1 weights FROZEN, + LoRMA)
#               -> token vectors standardized
#               -> [attention-pooled ; mean] of its token vectors = chunk vector
#   chunks of a case -> LayerNorm -> BiLSTM over the chunks (in order)
#                    -> attention pooling (learns WHICH chunks matter)
#                    -> 2 classes
#
# Layer names match the Architecture-3 notebook, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-3 E-LegalFormer (causal decoder, GQA, RoPE)
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_float = x.float()                                   # float32 for safety under float16
        x_float = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * self.weight).to(x.dtype)


class RotaryEmbedding(nn.Module):

    def __init__(self, dim, max_seq_len, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        freqs = torch.outer(torch.arange(max_seq_len), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, seq_len):
        return self.cos[:seq_len], self.sin[:seq_len]


def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    cos = torch.cat([cos, cos], dim=-1)[None, None].to(q.dtype)
    sin = torch.cat([sin, sin], dim=-1)[None, None].to(q.dtype)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


class GroupedQueryAttention(nn.Module):
    """12 query heads share 4 key/value heads. Causal, as in pretraining."""

    def __init__(self, cfg):
        super().__init__()
        self.n_q_heads = cfg["n_q_heads"]
        self.n_kv_heads = cfg["n_kv_heads"]
        self.head_dim = cfg["d_model"] // cfg["n_q_heads"]
        self.q_proj = nn.Linear(cfg["d_model"], self.n_q_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)

        repeat = self.n_q_heads // self.n_kv_heads
        k = k.repeat_interleave(repeat, dim=1)
        v = v.repeat_interleave(repeat, dim=1)

        # causal; padding is at the END, so real tokens never see it
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class SwiGLU(nn.Module):

    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ELegalFormerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attention = GroupedQueryAttention(cfg)
        self.ffn_norm = RMSNorm(cfg["d_model"])
        self.ffn = SwiGLU(cfg["d_model"], cfg["ffn_dim"])

    def forward(self, x, cos, sin):
        x = x + self.attention(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class ELegalFormer(nn.Module):
    """Architecture-3 without the LM head: tokens -> hidden vectors."""

    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.rope = RotaryEmbedding(cfg["d_model"] // cfg["n_q_heads"], cfg["max_seq_len"], cfg.get("rope_theta", 10000.0))
        self.layers = nn.ModuleList([ELegalFormerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["d_model"])

    def forward(self, input_ids, mask=None):
        """`mask` is not needed: attention is causal and padding is at the end."""
        x = self.token_embedding(input_ids)
        cos, sin = self.rope(input_ids.size(1))
        for layer in self.layers:
            x = layer(x, cos, sin)
        return self.final_norm(x)


# ------------------------------------------------------------
# LoRMA (multiplicative low-rank adaptation)
#     W' x = (I + alpha/r * B A) W x = h + alpha/r * B (A h),   h = W x
# A: out_features -> r,  B: r -> out_features,  B = 0 at the start (W' = W).
# A bias (only Architecture-2 has them) is kept as it is: y = W' x + b
# ------------------------------------------------------------

class LoRMALinear(nn.Module):

    def __init__(self, base, rank, alpha, dropout):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        out_features = base.out_features

        self.lorma_A = nn.Linear(out_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / rank

        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)                   # starts as the identity -> W' = W

    def forward(self, x):

        h = F.linear(x, self.base.weight)                                       # W x
        h = h + self.scale * self.lorma_B(self.lorma_A(self.dropout(h)))       # (I + alpha/r * B A) W x

        if self.base.bias is not None:
            h = h + self.base.bias.to(h.dtype)                                  # + b (frozen)

        return h


def add_peft(backbone):
    """Freeze every Phase-1 weight, then wrap the target linear layers of every block with LoRMA."""

    for p in backbone.parameters():
        p.requires_grad = False

    for block in backbone.layers:
        for target in LORMA_TARGETS:                          # e.g. "attention.q_proj"

            parent_name, name = target.rsplit(".", 1)
            parent = block.get_submodule(parent_name)

            setattr(parent, name, LoRMALinear(getattr(parent, name), LORMA_RANK, LORMA_ALPHA, LORMA_DROPOUT))


# ============================================================
# Standardization with RUNNING statistics (no learned weights)
#   x -> (x - mean) / std
# The statistics start from the frozen Phase-1 model (CELL 12) and then
# slowly follow the features while LoRMA changes them (fixed statistics
# would make the features drift away from what the head has learned).
# ============================================================

class RunningStandardize(nn.Module):

    def __init__(self, dim, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("std", torch.ones(dim))

    def forward(self, x, mask):
        """x [..., dim], mask [...] (True = real value)."""

        if self.training:
            with torch.no_grad():
                values = x[mask]
                if len(values) > 1:
                    self.mean.mul_(1 - self.momentum).add_(self.momentum * values.mean(dim=0))
                    self.std.mul_(1 - self.momentum).add_(self.momentum * values.std(dim=0).clamp(min=1e-4))

        return (x - self.mean) / self.std


# ============================================================
# Case classifier (hierarchical: tokens -> chunks -> case)
# ============================================================

class CaseClassifier(nn.Module):

    def __init__(self, backbone, d_model):
        super().__init__()

        self.backbone = backbone

        # Token vectors of a language model share one big direction (chunks of
        # different cases have cosine ~0.97); what tells cases apart is small.
        # Standardizing every feature (minus its mean over train tokens, divided
        # by its std) keeps only that part. Statistics: CELL 12.
        self.token_standardize = RunningStandardize(d_model)

        # token attention: LEARNS which tokens of a chunk matter (an average over
        # 512 tokens drowns the few words that decide a case)
        self.token_attention = nn.Sequential(nn.Linear(d_model, 128), nn.Tanh(), nn.Linear(128, 1))

        # chunk vectors are standardized again (statistics over train chunks, CELL 12):
        # this is what lets the head see the small differences between cases
        self.chunk_standardize = RunningStandardize(2 * d_model)

        self.vector_norm = nn.LayerNorm(2 * d_model)                  # [attention-pooled ; mean] -> 2 * d_model
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        self.lstm = nn.LSTM(2 * d_model, LSTM_HIDDEN, batch_first=True, bidirectional=True)

        self.attention = nn.Sequential(nn.Linear(2 * LSTM_HIDDEN, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Linear(2 * LSTM_HIDDEN, 2)

        self.register_buffer("class_weights", CLASS_WEIGHTS.clone().float())

    def peft_parameters(self):
        return [p for p in self.backbone.parameters() if p.requires_grad]

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def chunk_vectors(self, chunks):
        """List of token-id arrays -> [n_chunks, 2D] float32.
        Chunk vector = [attention-pooled ; mean] of its standardized token vectors."""

        max_len = max(len(c) for c in chunks)
        ids = torch.full((len(chunks), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(chunks), max_len), dtype=torch.bool)

        for row, c in enumerate(chunks):
            ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
            mask[row, :len(c)] = True

        ids, mask = ids.to(DEVICE), mask.to(DEVICE)

        with autocast():
            hidden = self.backbone(ids, mask)                         # [n_chunks, T, D]

        # do not pool the first POOL_SKIP_TOKENS tokens (unless the chunk is very short)
        pool_mask = mask.clone()
        long_chunk = mask.sum(dim=1) > 2 * POOL_SKIP_TOKENS
        pool_mask[long_chunk, :POOL_SKIP_TOKENS] = False

        z = self.token_standardize(hidden.float(), pool_mask)          # standardized token vectors

        scores = self.token_attention(z).squeeze(-1).masked_fill(~pool_mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)                         # [n_chunks, T]
        attended = (weights.unsqueeze(-1) * z).sum(dim=1)

        m = pool_mask.unsqueeze(-1)
        mean = (z * m).sum(dim=1) / m.sum(dim=1)

        return torch.cat([attended, mean], dim=-1)

    def classify(self, case_vectors):
        """
        case_vectors: list of [n_chunks_i, 2D] tensors (one per case, chunks in document order)
        returns logits [n_cases, 2] and attention weights [n_cases, max_chunks]
        """

        n_chunks = [len(v) for v in case_vectors]
        K = max(n_chunks)

        x = torch.zeros(len(case_vectors), K, case_vectors[0].size(-1), device=DEVICE)
        mask = torch.zeros(len(case_vectors), K, dtype=torch.bool, device=DEVICE)

        for i, v in enumerate(case_vectors):
            x[i, :len(v)] = v.to(DEVICE)
            mask[i, :len(v)] = True

        x = self.chunk_standardize(x, mask)
        x = self.dropout(self.vector_norm(x))

        lengths = torch.tensor(n_chunks)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=K)    # [n, K, 2H]

        scores = self.attention(out).squeeze(-1).masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)                                            # [n, K]

        case_vector = (weights.unsqueeze(-1) * out).sum(dim=1)

        return self.classifier(self.dropout(case_vector)), weights

    def forward(self, cases):

        all_chunks = [c for case in cases for c in case["chunks"]]
        vectors = self.chunk_vectors(all_chunks)

        case_vectors, k = [], 0
        for case in cases:
            n = len(case["chunks"])
            case_vectors.append(vectors[k:k + n])
            k += n

        return self.classify(case_vectors)

    def loss(self, cases):

        logits, _ = self(cases)
        labels = torch.tensor([case["label"] for case in cases], device=DEVICE)

        return F.cross_entropy(logits, labels, weight=self.class_weights)


# ------------------------------------------------------------
# Check: LoRMA really computes (I + alpha/r * B A) W x + b
# ------------------------------------------------------------

_layer = LoRMALinear(nn.Linear(8, 6, bias=True), rank=2, alpha=4, dropout=0.0)
nn.init.normal_(_layer.lorma_B.weight)
_x = torch.randn(3, 8)

_merged = (torch.eye(6) + _layer.scale * _layer.lorma_B.weight @ _layer.lorma_A.weight) @ _layer.base.weight

assert torch.allclose(_layer(_x), _x @ _merged.T + _layer.base.bias, atol=1e-5)
print("✅ LoRMA layer = (I + alpha/r * B A) W x + b")


✅ LoRMA layer = (I + alpha/r * B A) W x + b


In [10]:
# ============================================================
# CELL 10 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-3 CHECKPOINT
# ============================================================

def is_architecture3(path):

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ckpt, dict) and "n_kv_heads" in ckpt.get("config", {})
    except Exception:
        return False


if not CHECKPOINT_PATH.exists():

    found = [p for p in sorted(Path("/kaggle/input").rglob("best_model.pt")) if is_architecture3(p)]
    assert found, f"{CHECKPOINT_PATH} not found and no Architecture-3 checkpoint under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
MODEL_CONFIG = dict(phase1["config"])

# weights without the LM head (tied to the embedding, not needed here)
MODEL_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

del phase1

D_MODEL = MODEL_CONFIG["d_model"]

print("Phase-1 config:", MODEL_CONFIG)

assert MODEL_CONFIG["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert CHUNK_LEN <= MODEL_CONFIG["max_seq_len"], f"CHUNK_LEN must be <= {MODEL_CONFIG['max_seq_len']}"


def build_model():
    """Frozen Phase-1 E-LegalFormer + LoRMA + new case classifier."""

    backbone = ELegalFormer(MODEL_CONFIG)
    backbone.load_state_dict(MODEL_WEIGHTS, strict=True)          # every name and shape must match

    # LoRMA starts as "no change": the output must stay the same
    test_ids = torch.randint(4, VOCAB_SIZE, (2, 24))
    test_mask = torch.ones(2, 24, dtype=torch.bool)
    test_mask[1, 16:] = False                                    # second row: padding at the end

    backbone.eval()
    with torch.no_grad():
        before = backbone(test_ids, test_mask)
        add_peft(backbone)
        after = backbone(test_ids, test_mask)
    backbone.train()

    assert torch.allclose(before, after, atol=1e-4), "LoRMA changed the model at the start"

    return CaseClassifier(backbone, D_MODEL).to(DEVICE)


# free GPU memory if this cell is re-run
for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

trainable_names = [name for name, p in model.backbone.named_parameters() if p.requires_grad]
assert trainable_names and all("lorma_" in name for name in trainable_names), "only LoRMA matrices may train in the backbone"

n_total = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_peft = sum(p.numel() for p in model.peft_parameters())
n_head = sum(p.numel() for p in model.head_parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

assert n_trainable == n_peft + n_head

print(f"\nPhase-1 model (frozen)   : {n_backbone - n_peft:,}")
print(f"LoRMA (trainable)        : {n_peft:,}  ({n_peft / (n_backbone - n_peft):.3%} of the Phase-1 model, {len(trainable_names)} tensors)")
print(f"Head (trainable)         : {n_head:,}")
print(f"Trainable total          : {n_trainable:,}  ({n_trainable / n_total:.2%} of {n_total:,})")


Phase-1 config: {'vocab_size': 32004, 'max_seq_len': 2048, 'd_model': 768, 'n_layers': 16, 'n_q_heads': 12, 'n_kv_heads': 4, 'ffn_dim': 2048, 'dropout': 0.0, 'rope_theta': 10000.0}

Phase-1 model (frozen)   : 125,267,712
LoRMA (trainable)        : 3,538,944  (2.825% of the Phase-1 model, 224 tensors)
Head (trainable)         : 3,842,564
Trainable total          : 7,381,508  (5.56% of 132,649,220)


In [11]:
# ============================================================
# CELL 11 — PREDICTION + EVALUATION
#
# Chunk vectors are computed once per case (in big batches), then
# the classifier turns them into P(label = 1).
# ============================================================

@torch.no_grad()
def case_vectors(model, cases):
    """List of [n_chunks, 2D] CPU tensors, one per case."""

    model.eval()

    all_chunks = [c for case in cases for c in case["chunks"]]
    vectors = []

    for k in range(0, len(all_chunks), EVAL_CHUNKS):
        vectors.append(model.chunk_vectors(all_chunks[k:k + EVAL_CHUNKS]).cpu())

    vectors = torch.cat(vectors)

    out, k = [], 0
    for case in cases:
        n = len(case["chunks"])
        out.append(vectors[k:k + n])
        k += n

    model.train()

    return out


@torch.no_grad()
def classify_vectors(model, vectors, batch=256):
    """Chunk vectors -> (P(label = 1) [n_cases], attention weights list)."""

    model.eval()

    probs, weights = [], []

    for k in range(0, len(vectors), batch):

        logits, w = model.classify(vectors[k:k + batch])

        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
        weights.extend([w[i, :len(v)].cpu().numpy() for i, v in enumerate(vectors[k:k + batch])])

    model.train()

    return torch.cat(probs).numpy(), weights


def evaluate(model, cases):
    """Dev scores at threshold 0.5 and at the best threshold."""

    labels = np.array([c["label"] for c in cases])
    probs, _ = classify_vectors(model, case_vectors(model, cases))

    threshold, _ = best_threshold(labels, probs)

    return {
        "at_0.5": binary_scores(labels, (probs >= 0.5).astype(int)),
        "threshold": threshold,
        "at_threshold": binary_scores(labels, (probs >= threshold).astype(int)),
        "auc": float(roc_auc_score(labels, probs)) if len(set(labels)) == 2 else 0.0,
    }, probs

In [12]:
# ============================================================
# CELL 12 — PHASE-1 FEATURES: TOKEN STATISTICS + LINEAR PROBE (before training)
#
#   1. token vectors of STATS_DOCS train cases from the FROZEN Phase-1
#      model -> mean and std of every feature -> stored in the model;
#      then the same for the chunk vectors
#   2. how similar are chunks of different cases? (raw / standardized)
#   3. linear probe: logistic regression on the case vector (mean of the
#      standardized token vectors) -> dev AUC / macro-F1
#      = how much the frozen Phase-1 features know (for the report)
# ============================================================

from sklearn.linear_model import LogisticRegression

t0 = time.time()

# ---------------- 1. token statistics ----------------
stat_cases = random.Random(SEED).sample(train_cases, min(STATS_DOCS, len(train_cases)))
stat_chunks = [c for case in stat_cases for c in case["chunks"]]

total = torch.zeros(D_MODEL, dtype=torch.float64, device=DEVICE)
total_sq = torch.zeros_like(total)
count = 0
raw_chunk_means = []

model.eval()

with torch.no_grad():

    for k in range(0, len(stat_chunks), EVAL_CHUNKS):

        batch = stat_chunks[k:k + EVAL_CHUNKS]
        max_len = max(len(c) for c in batch)
        ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(batch), max_len), dtype=torch.bool)

        for row, c in enumerate(batch):
            ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
            mask[row, :len(c)] = True
            if len(c) > 2 * POOL_SKIP_TOKENS:
                mask[row, :POOL_SKIP_TOKENS] = False

        ids, mask = ids.to(DEVICE), mask.to(DEVICE)

        with autocast():
            hidden = model.backbone(ids, mask).float()

        m = mask.unsqueeze(-1)
        total += (hidden * m).sum(dim=(0, 1)).double()
        total_sq += (hidden.pow(2) * m).sum(dim=(0, 1)).double()
        count += int(mask.sum())

        raw_chunk_means.append(((hidden * m).sum(dim=1) / m.sum(dim=1)).cpu())

model.train()

TOKEN_MEAN = (total / count).float()
TOKEN_STD = (total_sq / count - (total / count).pow(2)).clamp(min=1e-8).sqrt().float().clamp(min=1e-4)

model.token_standardize.mean.copy_(TOKEN_MEAN)
model.token_standardize.std.copy_(TOKEN_STD)

print(f"Token statistics from {count:,} tokens of {len(stat_cases)} train cases in {time.time() - t0:.0f}s")

# chunk statistics (same cases, now with standardized tokens)
stat_vectors = torch.cat(case_vectors(model, stat_cases))

FEATURE_MEAN = stat_vectors.mean(dim=0)
FEATURE_STD = stat_vectors.std(dim=0).clamp(min=1e-4)

model.chunk_standardize.mean.copy_(FEATURE_MEAN.to(DEVICE))
model.chunk_standardize.std.copy_(FEATURE_STD.to(DEVICE))


# ---------------- 2. how similar are chunks of DIFFERENT cases? ----------------
raw = torch.cat(raw_chunk_means)                                   # mean of raw token vectors
std = (stat_vectors - FEATURE_MEAN) / FEATURE_STD                  # what the head sees

_i = torch.randint(0, len(raw), (1000,))
_j = torch.randint(0, len(raw), (1000,))
cos_raw = F.cosine_similarity(raw[_i], raw[_j], dim=-1).mean().item()
cos_std = F.cosine_similarity(std[_i], std[_j], dim=-1).mean().item()

print(f"mean cosine between random chunks: raw {cos_raw:.3f} -> standardized {cos_std:.3f}")


# ---------------- 3. linear probe on the frozen features ----------------
def case_matrix(vectors):
    # case vector = mean over chunks of the standardized MEAN half of the chunk vectors (no learned weights)
    d = D_MODEL
    return np.stack([((v - FEATURE_MEAN) / FEATURE_STD)[:, d:].mean(dim=0).numpy() for v in vectors])


probe_cases = random.Random(SEED + 1).sample(train_cases, min(PROBE_DOCS, len(train_cases)))

X_probe = case_matrix(case_vectors(model, probe_cases))
y_probe = np.array([c["label"] for c in probe_cases])

X_dev = case_matrix(case_vectors(model, dev_cases))
y_dev = np.array([c["label"] for c in dev_cases])

probe_results = {}

for C in (0.01, 0.1, 1.0):

    clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
    clf.fit(X_probe, y_probe)
    p_dev = clf.predict_proba(X_dev)[:, 1]

    t, f1 = best_threshold(y_dev, p_dev)
    probe_results[C] = {"auc": float(roc_auc_score(y_dev, p_dev)), "f1": f1, "threshold": t}

    print(f"  probe C={C:<5} | dev AUC {probe_results[C]['auc']:.4f} | macro-F1 {f1:.4f} (threshold {t:.2f})")

PROBE_BEST = max(probe_results.values(), key=lambda r: r["auc"])

print(f"\nFrozen Phase-1 features (mean of tokens) + logistic regression: dev AUC {PROBE_BEST['auc']:.4f}, "
      f"macro-F1 {PROBE_BEST['f1']:.4f} | {time.time() - t0:.0f}s")

del raw, std, raw_chunk_means, stat_vectors, X_probe, X_dev

Token statistics from 1,423,202 tokens of 1000 train cases in 35s
mean cosine between random chunks: raw 0.724 -> standardized 0.013
  probe C=0.01  | dev AUC 0.5981 | macro-F1 0.5768 (threshold 0.51)
  probe C=0.1   | dev AUC 0.5807 | macro-F1 0.5653 (threshold 0.48)
  probe C=1.0   | dev AUC 0.5689 | macro-F1 0.5530 (threshold 0.39)

Frozen Phase-1 features (mean of tokens) + logistic regression: dev AUC 0.5981, macro-F1 0.5768 | 331s


In [13]:
# ============================================================
# CELL 13 — OPTIMIZER + TRAINING FUNCTION
#   group 0 = LoRMA, group 1 = head
# ============================================================

def make_optimizer(model, total_steps, head_only_steps):

    optimizer = torch.optim.AdamW(
        [
            {"params": model.peft_parameters(), "lr": LR_PEFT, "weight_decay": 0.0},
            {"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.01},
        ],
        betas=(0.9, 0.999),
    )

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0. LoRMA waits for head_only_steps."""

        if step < warmup_steps:
            factor = (step + 1) / warmup_steps
        else:
            factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        for group, base_lr in zip(optimizer.param_groups, base_lrs):
            group["lr"] = base_lr * factor

        if step < head_only_steps:
            optimizer.param_groups[0]["lr"] = 0.0

    return optimizer, set_lr


def trainable_state(model):
    """The trained weights (LoRMA + head) + the running feature statistics: small, quick to copy."""

    state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
    state.update({name: b.detach().cpu().clone() for name, b in model.named_buffers() if "standardize" in name})

    return state


def train_model(model, epochs):
    """
    Trains on TRAIN cases only. Every epoch uses EPOCH_DOCS cases (the train set is
    shuffled once and walked through, so every case is used before any repeats).
    After every epoch: dev macro-F1 (at the best threshold); the best epoch is kept.
    """

    docs_per_epoch = min(EPOCH_DOCS, len(train_cases))
    updates_per_epoch = math.ceil(docs_per_epoch / (BATCH_DOCS * GRAD_ACCUM))

    optimizer, set_lr = make_optimizer(model, epochs * updates_per_epoch, HEAD_ONLY_EPOCHS * updates_per_epoch)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    rng = random.Random(SEED)
    order = list(range(len(train_cases)))
    rng.shuffle(order)
    cursor = 0

    history = []
    best_score, best_epoch, best_state = -1.0, 0, None
    step = 0
    t_start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        # the next docs_per_epoch cases of the shuffled order
        if cursor + docs_per_epoch > len(order):
            rng.shuffle(order)
            cursor = 0

        epoch_cases = [train_cases[i] for i in order[cursor:cursor + docs_per_epoch]]
        cursor += docs_per_epoch

        batches = [epoch_cases[k:k + BATCH_DOCS] for k in range(0, len(epoch_cases), BATCH_DOCS)]

        total_loss, skipped = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b, batch in enumerate(batches):

            loss = model.loss(batch)
            scaler.scale(loss / GRAD_ACCUM).backward()
            total_loss += loss.item()

            if (b + 1) % GRAD_ACCUM == 0 or b + 1 == len(batches):

                set_lr(step)

                scaler.unscale_(optimizer)

                # clip LoRMA and head separately
                torch.nn.utils.clip_grad_norm_(optimizer.param_groups[0]["params"], GRAD_CLIP)
                torch.nn.utils.clip_grad_norm_(optimizer.param_groups[1]["params"], GRAD_CLIP)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += scaler.get_scale() < scale_before

                optimizer.zero_grad(set_to_none=True)
                step += 1

            if (b + 1) % 500 == 0:
                print(f"    batch {b + 1}/{len(batches)} | loss {total_loss / (b + 1):.4f} | "
                      f"{(time.time() - t_start) / 60:.1f} min", flush=True)

        dev_scores, _ = evaluate(model, dev_cases)

        log = {"epoch": epoch, "loss": total_loss / len(batches), "dev": dev_scores, "skipped_steps": int(skipped)}

        line = (f"Epoch {epoch}/{epochs}{' (head only)' if epoch <= HEAD_ONLY_EPOCHS else ''} | loss {log['loss']:.4f} | "
                f"dev macro-F1 @0.5 {dev_scores['at_0.5']['f1']:.4f} | "
                f"@{dev_scores['threshold']:.2f} {dev_scores['at_threshold']['f1']:.4f} | AUC {dev_scores['auc']:.4f}")

        stop = False

        if dev_scores["at_threshold"]["f1"] > best_score:
            best_score, best_epoch, best_state = dev_scores["at_threshold"]["f1"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        line += f" | {(time.time() - t_start) / 60:.1f} min"

        print(line, flush=True)
        history.append(log)

        if epoch == 1 and dev_scores["auc"] < 0.53:
            print("WARNING: dev AUC < 0.53 after epoch 1 -> the head is not learning. "
                  "Stop the run and send me CELL 12 and this log.")

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state, strict=False)          # back to the best epoch

    return history, best_epoch

In [14]:
# ============================================================
# CELL 14 — TRAIN (TRAIN data only; dev picks the best epoch)
# ============================================================

print(f"{RUN_NAME} | train {len(train_cases)} cases ({min(EPOCH_DOCS, len(train_cases))} per epoch) "
      f"-> dev {len(dev_cases)} cases")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
history, best_epoch = train_model(model, EPOCHS)
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

print(f"\n✅ Trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")

p4_arch3_lorma | train 37387 cases (8000 per epoch) -> dev 3505 cases
    batch 500/2000 | loss 0.6941 | 3.7 min
    batch 1000/2000 | loss 0.6935 | 7.3 min
    batch 1500/2000 | loss 0.6774 | 10.9 min
    batch 2000/2000 | loss 0.6647 | 14.6 min
Epoch 1/10 (head only) | loss 0.6647 | dev macro-F1 @0.5 0.5326 | @0.40 0.5851 | AUC 0.6191  <- best | 16.9 min
    batch 500/2000 | loss 0.6347 | 20.6 min
    batch 1000/2000 | loss 0.6230 | 24.3 min
    batch 1500/2000 | loss 0.6157 | 28.0 min
    batch 2000/2000 | loss 0.6056 | 31.7 min
Epoch 2/10 | loss 0.6056 | dev macro-F1 @0.5 0.5896 | @0.35 0.6528 | AUC 0.7131  <- best | 34.1 min
    batch 500/2000 | loss 0.5603 | 37.8 min
    batch 1000/2000 | loss 0.5646 | 41.4 min
    batch 1500/2000 | loss 0.5620 | 45.1 min
    batch 2000/2000 | loss 0.5611 | 48.8 min
Epoch 3/10 | loss 0.5611 | dev macro-F1 @0.5 0.5556 | @0.22 0.6671 | AUC 0.7318  <- best | 51.2 min
    batch 500/2000 | loss 0.5399 | 54.9 min
    batch 1000/2000 | loss 0.5370 | 58.

In [15]:
# ============================================================
# CELL 15 — DEV REPORT
#   scores + threshold, confusion matrix, errors by document length,
#   which chunks the model uses, false positives / negatives
# ============================================================

t0 = time.time()
dev_vectors = case_vectors(model, dev_cases)
dev_probs, dev_attention = classify_vectors(model, dev_vectors)
predict_seconds = time.time() - t0

dev_labels = np.array([c["label"] for c in dev_cases])

THRESHOLD, _ = best_threshold(dev_labels, dev_probs)        # used for the test predictions (CELL 16)
dev_preds = (dev_probs >= THRESHOLD).astype(int)

at_05 = binary_scores(dev_labels, (dev_probs >= 0.5).astype(int))
at_t = binary_scores(dev_labels, dev_preds)
auc = float(roc_auc_score(dev_labels, dev_probs))

print(f"DEV ({RUN_NAME}, best epoch {best_epoch})")
print(f"{'':16s} {'macro-P':>8s} {'macro-R':>8s} {'macro-F1':>9s}")
print(f"{'threshold 0.50':16s} {at_05['precision']:8.4f} {at_05['recall']:8.4f} {at_05['f1']:9.4f}")
print(f"{f'threshold {THRESHOLD:.2f}':16s} {at_t['precision']:8.4f} {at_t['recall']:8.4f} {at_t['f1']:9.4f}")
print(f"ROC-AUC {auc:.4f} | predicted in {predict_seconds:.0f}s")

p, r, f, support = precision_recall_fscore_support(dev_labels, dev_preds, labels=[0, 1], zero_division=0)
print(f"\nper class      P      R      F1   gold")
for k in (0, 1):
    print(f"  label {k}  {p[k]:6.3f} {r[k]:6.3f} {f[k]:6.3f} {support[k]:6d}")

cm = confusion_matrix(dev_labels, dev_preds, labels=[0, 1])
print(f"\nconfusion matrix (rows = gold, columns = predicted)\n          pred 0  pred 1\n  gold 0  {cm[0, 0]:6d}  {cm[0, 1]:6d}\n  gold 1  {cm[1, 0]:6d}  {cm[1, 1]:6d}")


# ------------------------------------------------------------
# Errors by document length (characters), in 4 equal groups
# ------------------------------------------------------------

chars = np.array([len(c["text"]) for c in dev_cases])
edges = np.percentile(chars, [0, 25, 50, 75, 100])

print(f"\n{'length (chars)':>22s} {'cases':>6s} {'macro-F1':>9s} {'false +':>8s} {'false -':>8s}")

length_groups = []

for g in range(4):

    sel = (chars >= edges[g]) & ((chars <= edges[g + 1]) if g == 3 else (chars < edges[g + 1]))

    if sel.sum() == 0:
        continue

    s = binary_scores(dev_labels[sel], dev_preds[sel])
    fp = int(((dev_preds == 1) & (dev_labels == 0) & sel).sum())
    fn = int(((dev_preds == 0) & (dev_labels == 1) & sel).sum())

    length_groups.append({"min_chars": int(edges[g]), "max_chars": int(edges[g + 1]), "cases": int(sel.sum()),
                          "f1": s["f1"], "false_pos": fp, "false_neg": fn})

    print(f"{int(edges[g]):>10d} - {int(edges[g + 1]):>9d} {int(sel.sum()):6d} {s['f1']:9.4f} {fp:8d} {fn:8d}")


# ------------------------------------------------------------
# Which chunks matter?
#   1. average attention weight per chunk position (counted from the end
#      for DOC_PART "end", from the start otherwise)
#   2. macro-F1 when the classifier sees ONLY one chunk position
# ------------------------------------------------------------

from_end = DOC_PART == "end"

attention_by_position = {}
for w in dev_attention:
    for k, value in enumerate(w[::-1] if from_end else w):
        attention_by_position.setdefault(k, []).append(float(value))

single_chunk = {}

for k in range(max(len(v) for v in dev_vectors)):

    subset, idx = [], []

    for i, v in enumerate(dev_vectors):
        if len(v) > k:
            subset.append(v[len(v) - 1 - k:len(v) - k] if from_end else v[k:k + 1])
            idx.append(i)

    if not subset:
        continue

    probs_k, _ = classify_vectors(model, subset)
    single_chunk[k] = {"cases": len(idx), "f1": binary_scores(dev_labels[idx], (probs_k >= THRESHOLD).astype(int))["f1"]}

name = "chunk (0 = last)" if from_end else "chunk (0 = first)"

if DOC_PART == "start_end":
    print("(start_end: the first chunks come from the start of the document, the last ones from its end)")
print(f"\n{name:18s} {'mean attention':>15s} {'F1 with only this chunk':>24s}")

for k in sorted(attention_by_position):
    print(f"{k:18d} {np.mean(attention_by_position[k]):15.3f} {single_chunk.get(k, {}).get('f1', 0):24.4f}")

print(f"{'all chunks':18s} {'':15s} {at_t['f1']:24.4f}")


# ------------------------------------------------------------
# False positives / negatives: saved for reading
# ------------------------------------------------------------

errors = []

for case, label, pred, prob in zip(dev_cases, dev_labels, dev_preds, dev_probs):
    if label != pred:
        errors.append({"id": case["id"], "gold": int(label), "pred": int(pred), "prob_1": round(float(prob), 4),
                       "type": "false positive" if pred == 1 else "false negative",
                       "chars": len(case["text"]), "end_of_text": case["text"][-600:]})

fp_chars = [e["chars"] for e in errors if e["type"] == "false positive"]
fn_chars = [e["chars"] for e in errors if e["type"] == "false negative"]
ok_chars = [len(c["text"]) for c, l, p in zip(dev_cases, dev_labels, dev_preds) if l == p]

print(f"\nmedian length (chars): correct {np.median(ok_chars):.0f} | "
      f"false positives {np.median(fp_chars) if fp_chars else 0:.0f} ({len(fp_chars)}) | "
      f"false negatives {np.median(fn_chars) if fn_chars else 0:.0f} ({len(fn_chars)})")

with open(OUTPUT_DIR / "dev_errors.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=1)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "Architecture-3 E-LegalFormer (frozen) + LoRMA, token attention + chunk BiLSTM/attention classifier",
    "trained_on": "train only (dev used to pick the best epoch and the threshold)",
    "doc_part": DOC_PART, "doc_tokens": DOC_TOKENS, "chunk_len": CHUNK_LEN,
    "peft": {"method": "LoRMA", "rank": LORMA_RANK, "alpha": LORMA_ALPHA, "targets": LORMA_TARGETS},
    "parameters_total": n_total, "parameters_trainable": n_trainable,
    "parameters_peft": n_peft, "parameters_head": n_head,
    "best_epoch": best_epoch, "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "threshold": THRESHOLD,
    "probe_frozen_features": {"auc": PROBE_BEST["auc"], "f1": PROBE_BEST["f1"], "by_C": probe_results,
                              "cosine_raw": cos_raw, "cosine_standardized": cos_std},
    "signal_check_tfidf": signal_check,
    "dev_at_0.5": at_05, "dev_at_threshold": at_t, "dev_auc": auc,
    "dev_confusion_matrix": cm.tolist(),
    "dev_by_length": length_groups,
    "attention_by_chunk": {k: float(np.mean(v)) for k, v in attention_by_position.items()},
    "f1_single_chunk": single_chunk,
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / "dev_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(dev_cases, dev_probs)}, f)

print("\nSaved:", OUTPUT_DIR / "results.json", ", dev_errors.json, dev_probabilities.json")

DEV (p4_arch3_lorma, best epoch 6)
                  macro-P  macro-R  macro-F1
threshold 0.50     0.6996   0.6598    0.6416
threshold 0.33     0.6850   0.6823    0.6811
ROC-AUC 0.7462 | predicted in 145s

per class      P      R      F1   gold
  label 0   0.662  0.743  0.700   1749
  label 1   0.708  0.622  0.662   1756

confusion matrix (rows = gold, columns = predicted)
          pred 0  pred 1
  gold 0    1299     450
  gold 1     664    1092

        length (chars)  cases  macro-F1  false +  false -
       434 -     10803    875    0.6870      110      163
     10803 -     16988    877    0.6742      111      174
     16988 -     26860    875    0.6922      107      160
     26860 -    260198    878    0.6704      122      167

chunk (0 = last)    mean attention  F1 with only this chunk
                 0           0.465                   0.6343
                 1           0.281                   0.5675
                 2           0.258                   0.5928
all chunks       

In [16]:
# ============================================================
# CELL 16 — PREDICT TEST + WRITE prediction.json + submission.zip
#   format: [{"id": "case_001", "pred": 1}, ...]
# ============================================================

t0 = time.time()
test_probs, _ = classify_vectors(model, case_vectors(model, test_cases))
print(f"Predicted {len(test_cases)} test cases in {time.time() - t0:.0f}s (threshold {THRESHOLD:.2f})")

submission = [{"id": case["id"], "pred": int(prob >= THRESHOLD)} for case, prob in zip(test_cases, test_probs)]


# ------------------------------------------------------------
# Format checks
# ------------------------------------------------------------

assert len(submission) == len(test_cases)
assert [s["id"] for s in submission] == [c["id"] for c in test_cases]
assert all(isinstance(s["pred"], int) and s["pred"] in (0, 1) for s in submission)

counts = Counter(s["pred"] for s in submission)
print(f"✅ {len(submission)} predictions | pred 0: {counts[0]} | pred 1: {counts[1]}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=1)

with open(OUTPUT_DIR / "test_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(test_cases, test_probs)}, f)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("submission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)

Predicted 1517 test cases in 62s (threshold 0.33)
✅ 1517 predictions | pred 0: 825 | pred 1: 692
submission.zip: ['prediction.json']
Saved: /kaggle/working/p4_arch3_lorma/submission.zip
